<a href="https://colab.research.google.com/github/goutham3010/Hospital-Patient-Flow-and-Resource-Demand-Prediction-System/blob/main/04_Model_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HOSPITAL PATIENT FLOW & DEMAND PREDICTION SYSTEM
## NOTEBOOK 04: MACHINE LEARNING MODEL TRAINING
**Workflow Phases 05, 06 & 07** of the 17-Stage Hospital Operations Decision Support Architecture.

### Objectives:
1. Implement a strict **Time-Based Train/Test Split** (< 2024 for training, 2024 for testing) to prevent temporal data leakage.
2. Train multiple regression architectures side-by-side:
   - **Linear Regression** (Parametric Baseline)
   - **Decision Tree Regressor** (Orthogonal Partitioning)
   - **Random Forest Regressor** (Bagging Ensemble)
   - **Gradient Boosting Regressor** (Boosting Ensemble)
3. Train specialized ward-level demand models for **General, ICU, HDU, and NICU** wards.
4. Benchmark out-of-sample error metrics on unseen future operational data.


In [ ]:
# 1. IMPORT LIBRARIES
import pandas as pd
import numpy as np
import sys
import os

sys.path.append('src')
from train import time_based_train_test_split, train_baseline_models, train_ward_demand_models
from preprocessing import get_feature_columns

print("Training modules loaded.")


In [ ]:
# 2. LOAD PREPROCESSED DAILY FLOW DATASET
df = pd.read_csv('data/daily_patient_flow.csv')
feature_cols = get_feature_columns()
target_col = 'target_tomorrow_admissions'

print(f"Dataset Loaded: {len(df)} days.")


In [ ]:
# 3. TIME-BASED TRAIN/TEST SPLIT (CRUCIAL FOR TIME SERIES)
# We partition chronologically: 2015-2023 for training, 2024 for testing.
train_df, test_df = time_based_train_test_split(df, split_date='2024-01-01')

X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test = test_df[feature_cols]
y_test = test_df[target_col]

print(f"Training Set Horizon: {len(train_df)} days ({train_df['date'].min()} to {train_df['date'].max()})")
print(f"Testing Set Horizon:  {len(test_df)} days ({test_df['date'].min()} to {test_df['date'].max()})")


In [ ]:
# 4. TRAIN AND EVALUATE MULTIPLE REGRESSION ARCHITECTURES
trained_models, model_metrics, model_predictions = train_baseline_models(
    X_train, y_train, X_test, y_test
)


In [ ]:
# 5. BENCHMARK COMPARISON TABLE
benchmark_df = pd.DataFrame(model_metrics).T
benchmark_df


In [ ]:
# 6. TRAIN DEPARTMENTAL WARD-SPECIFIC DEMAND MODELS
ward_models = train_ward_demand_models(train_df, feature_cols)
print(f"Ward models trained: {list(ward_models.keys())}")
